# Laya V2.8 — Fast Full-Attention-First PDelta3 + Local32

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V28_Fast_FullAttention_First_Colab.ipynb)

V2.8 deliberately does **one thing first**: replace **all 10 ModernBERT full-attention layers**
`[0, 3, 6, 9, 12, 15, 18, 21, 24, 27]` with the strongest V2.6/V2.7 candidate:

**PDelta3-GDN2 + learned global linear memory + exact bidirectional Local32**.

All 18 sliding-attention layers remain native in this version. This isolates whether full attention can be removed cleanly before touching the sliding layers.

### Why V2.8 is faster than V2.7

1. **One teacher pass captures all full layers.** V2.7 regenerated teacher I/O separately per layer. V2.8 hooks all 10 full-attention layers at once and stores their targets on CPU.
2. **Fast local schedule.** Each layer gets a 200-step screen and only 100 polish steps once it already passes the local gate. Difficult layers can continue up to 600 total steps.
3. **No 180-step end-to-end recovery after every layer.** All locally valid full replacements are installed first.
4. **One joint recovery** then trains the complete all-full replacement model against teacher decisions, decision-marker representations, and intermediate full-attention outputs.
5. The official test split remains untouched until the final evaluation.

The notebook is fail-closed: if a full-attention layer cannot meet the local transfer gate by the hard limit, V2.8 stops instead of pretending that all full attention was successfully replaced.


In [ ]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Imports, configuration, teacher/student
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds
from tinycenn_lm.decision_v28 import (
    make_v28_full_replacement,
    split_v28_parameters,
)
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import (
    require_finite, guarded_step, forward_with_markers,
    marker_alignment_loss, decision_distill_loss,
)

SEED = 2028
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}

TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 100 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}

# Local transfer is candidate-only, so batch 4 usually fits on a T4/A10/L4.
LOCAL_BATCH = 4 #@param {type:"integer"}
JOINT_BATCH = 1 #@param {type:"integer"}

FULL_FEATURE_DIM = 96 #@param {type:"integer"}
FULL_LOCAL_WINDOW = 32 #@param {type:"integer"}
FULL_CHUNK_SIZE = 32 #@param {type:"integer"}

# Fast local training: 200 screen + 100 polish for easy layers.
SCREEN_STEPS = 200 #@param {type:"integer"}
POLISH_STEPS = 100 #@param {type:"integer"}
HARD_MAX_STEPS = 600 #@param {type:"integer"}
LOCAL_CHECK_EVERY = 50 #@param {type:"integer"}

# One teacher forward captures all ten full-attention targets.
TRAIN_CACHE_BATCHES = 20 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 5 #@param {type:"integer"}

LOCAL_MAX_NMSE = 0.20 #@param {type:"number"}
LOCAL_MIN_COSINE = 0.90 #@param {type:"number"}

# Joint recovery of all full replacements.
JOINT_STEPS = 250 #@param {type:"integer"}
JOINT_HARD_MAX_STEPS = 400 #@param {type:"integer"}
JOINT_CHECK_EVERY = 25 #@param {type:"integer"}

DEV_GATE_LIMIT = 500 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT = 0.95 #@param {type:"number"}
MAX_MEAN_JS = 0.01 #@param {type:"number"}
MAX_GOLD_ACCURACY_DROP = 0.02 #@param {type:"number"}

OUTPUT_DIR = Path("/content/Laya_V28_Fast_FullAttention_First")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = OUTPUT_DIR / "v28_progress.pt"
REPORT_PATH = OUTPUT_DIR / "v28_report.json"
RESUME = True #@param {type:"boolean"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("V2.8 requires a CUDA GPU.")

device = torch.device("cuda")
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported(including_emulation=False)
    else torch.float16
)

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=[
        "rl_agent_config.json",
        "model.safetensors",
        "encoder/*",
        "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))

teacher.to(device).eval().requires_grad_(False)
student.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, k in enumerate(kinds) if k == "full_attention"]
sliding_layers = [i for i, k in enumerate(kinds) if k == "sliding_attention"]

print("device:", torch.cuda.get_device_name(0))
print("amp:", amp_dtype)
print("full layers to replace:", full_layers)
print("sliding layers kept native:", sliding_layers)
print(
    "V2.8 local schedule:",
    f"{SCREEN_STEPS} screen + {POLISH_STEPS} polish, hard max {HARD_MAX_STEPS}"
)


In [ ]:
#@title 3. Leak-free train/dev data
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon = [_canon_label(k) for k in labels]
    if gold_label not in canon:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer, state, q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

all_train_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
split_rng = random.Random(SEED)
split_rng.shuffle(all_train_rows)
assert TRAIN_CASES + DEV_CASES <= len(all_train_rows)

train_rows = all_train_rows[:TRAIN_CASES]
dev_rows = all_train_rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

print("train/dev decisions:", len(train_items), len(dev_items))
print("official test is deferred until the final evaluation")


In [ ]:
#@title 4. Fast multi-layer teacher cache + evaluation helpers
def _cpu(t):
    if t is None:
        return None
    return t.detach().to("cpu").contiguous()

def _cpu_pos(pos):
    if pos is None:
        return None
    return tuple(_cpu(z) for z in pos)

def _gpu(t):
    if t is None:
        return None
    return t.to(device, non_blocking=True)

def _gpu_pos(pos):
    if pos is None:
        return None
    return tuple(_gpu(z) for z in pos)

def cache_all_full_teacher_io(items, batches, batch_size, offset=0):
    """Capture every full-attention target in ONE teacher forward per batch."""
    cache = {i: [] for i in full_layers}
    captured = {}
    handles = []

    def make_pre(idx):
        def pre(mod, args, kwargs):
            captured[idx] = {
                "x": args[0].detach(),
                "pos": None if kwargs.get("position_embeddings") is None else tuple(
                    z.detach() for z in kwargs["position_embeddings"]
                ),
                "mask": None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach(),
            }
        return pre

    def make_post(idx):
        def post(mod, args, kwargs, out):
            captured[idx]["y"] = out[0].detach()
        return post

    for idx in full_layers:
        attn = teacher.encoder.layers[idx].attn
        handles.append(attn.register_forward_pre_hook(make_pre(idx), with_kwargs=True))
        handles.append(attn.register_forward_hook(make_post(idx), with_kwargs=True))

    try:
        teacher.eval()
        for bi in range(batches):
            b = batch(items, offset + bi * batch_size, batch_size)
            captured.clear()
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
                teacher(**b)
            valid = b["attention_mask"].bool()
            for idx in full_layers:
                got = captured[idx]
                cache[idx].append({
                    "x": _cpu(got["x"]),
                    "pos": _cpu_pos(got["pos"]),
                    "mask": _cpu(got["mask"]),
                    "y": _cpu(got["y"]),
                    "valid": _cpu(valid),
                })
            if (bi + 1) % 5 == 0 or bi + 1 == batches:
                print(f"  cached {bi+1}/{batches} batches for all {len(full_layers)} full layers")
    finally:
        for h in handles:
            h.remove()
    return cache

def promote_layer_cache(entries):
    return [{
        "x": _gpu(e["x"]),
        "pos": _gpu_pos(e["pos"]),
        "mask": _gpu(e["mask"]),
        "y": _gpu(e["y"]),
        "valid": _gpu(e["valid"]).bool(),
    } for e in entries]

@torch.no_grad()
def local_metrics(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    for e in cache:
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = repl(
                e["x"],
                position_embeddings=e["pos"],
                attention_mask=e["mask"],
            )
        require_finite(pred, "V2.8 local probe")
        valid = e["valid"]
        m = valid[:, :, None].float()
        diff_sum += float((((pred.float() - e["y"].float()) ** 2) * m).sum())
        power_sum += float(((e["y"].float() ** 2) * m).sum())
        cos_sum += float(F.cosine_similarity(
            pred.float()[valid],
            e["y"].float()[valid],
            dim=-1,
        ).sum())
        tokens += int(valid.sum())
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    return {
        "nmse": nmse,
        "cosine": cosine,
        "score": nmse + 1.10 * (1.0 - cosine),
        "tokens": tokens,
    }

@torch.no_grad()
def validate_pair(model, items=dev_items, limit=None, eval_batch=4):
    teacher.eval()
    model.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []

    for off in range(0, limit, eval_batch):
        selected = items[off:min(off + eval_batch, limit)]
        b = direct_batch(selected)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = model(**b)

        mask = b["marker_mask"].bool()
        tm = tl.float().masked_fill(~mask, -1e9)
        sm = sl.float().masked_fill(~mask, -1e9)
        tp = torch.softmax(tm, -1)
        sp = torch.softmax(sm, -1)
        tpred, spred = tm.argmax(-1), sm.argmax(-1)
        gold = torch.tensor(
            [item["gold_index"] for item in selected],
            device=device,
            dtype=torch.long,
        )

        agree += int((tpred == spred).sum())
        teacher_gold += int((tpred == gold).sum())
        student_gold += int((spred == gold).sum())
        total += len(selected)

        mid = 0.5 * (tp + sp)
        j = 0.5 * (
            tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())
        ).sum(-1)
        j += 0.5 * (
            sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())
        ).sum(-1)
        js.extend(j.cpu().tolist())

    return {
        "decisions": int(total),
        "agreement": agree / max(1, total),
        "mean_js": float(np.mean(js)),
        "teacher_gold_accuracy": teacher_gold / max(1, total),
        "student_gold_accuracy": student_gold / max(1, total),
        "accuracy_drop": (teacher_gold - student_gold) / max(1, total),
    }

def quality_pass(m):
    return bool(
        m["agreement"] >= MIN_TEACHER_AGREEMENT
        and m["mean_js"] <= MAX_MEAN_JS
        and m["accuracy_drop"] <= MAX_GOLD_ACCURACY_DROP
    )

def dev_rank(m):
    return (
        m["mean_js"]
        + 0.75 * (1.0 - m["agreement"])
        + 0.50 * max(0.0, m["accuracy_drop"])
    )


In [ ]:
#@title 5. Cache teacher targets once
print("Caching training targets for all full layers...")
train_cache_cpu = cache_all_full_teacher_io(
    train_items,
    TRAIN_CACHE_BATCHES,
    LOCAL_BATCH,
    offset=0,
)
print("Caching probe targets for all full layers...")
probe_cache_cpu = cache_all_full_teacher_io(
    dev_items,
    PROBE_CACHE_BATCHES,
    LOCAL_BATCH,
    offset=0,
)
print("✅ teacher cache ready")


In [ ]:
#@title 6. Fast local transfer of ALL full-attention layers
def state_cpu(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

def phase_lr(step, total, start, end):
    if total <= 1:
        return float(end)
    x = min(1.0, max(0.0, step / float(total - 1)))
    return float(end + 0.5 * (start - end) * (1 + math.cos(math.pi * x)))

def local_gate(m):
    return bool(m["nmse"] <= LOCAL_MAX_NMSE and m["cosine"] >= LOCAL_MIN_COSINE)

def train_local_fast(layer_idx, repl, train_cache, probe_cache):
    core, output = split_v28_parameters(repl)
    for p in core + output:
        if p.is_floating_point():
            p.data = p.data.float()

    opt = torch.optim.AdamW(
        [
            {"params": core, "lr": 1.0e-2, "weight_decay": 2e-4},
            {"params": output, "lr": 1.0e-3, "weight_decay": 1e-4},
        ],
        betas=(0.9, 0.95),
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=amp_dtype == torch.float16, init_scale=128.0
    )

    best = local_metrics(repl, probe_cache)
    best_state = state_cpu(repl)
    best_step = 0
    updates = attempts = 0
    target_steps = SCREEN_STEPS

    print(
        f"  start NMSE={best['nmse']:.4f} cos={best['cosine']:.4f}"
    )

    while updates < HARD_MAX_STEPS:
        # 0..SCREEN: 0.01 -> 0.002
        # after SCREEN: 0.002 -> 0.0005 over the remaining hard budget.
        if updates < SCREEN_STEPS:
            core_lr = phase_lr(updates, SCREEN_STEPS, 1.0e-2, 2.0e-3)
        else:
            core_lr = phase_lr(
                updates - SCREEN_STEPS,
                max(1, HARD_MAX_STEPS - SCREEN_STEPS),
                2.0e-3,
                5.0e-4,
            )
        opt.param_groups[0]["lr"] = core_lr
        opt.param_groups[1]["lr"] = core_lr * 0.10

        e = train_cache[attempts % len(train_cache)]
        repl.train()
        repl.out_drop.eval()
        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = repl(
                e["x"],
                position_embeddings=e["pos"],
                attention_mask=e["mask"],
            )
            valid = e["valid"]
            m = valid[:, :, None].float()
            mse = (
                ((pred.float() - e["y"].float()) ** 2) * m
            ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = (
                (e["y"].float() ** 2) * m
            ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cosine = F.cosine_similarity(
                pred.float()[valid], e["y"].float()[valid], dim=-1
            ).mean()
            loss = nmse + 1.10 * (1.0 - cosine)

        updated = guarded_step(
            loss,
            opt,
            scaler,
            [(core, 1.0), (output, 0.25)],
            context=f"V2.8 layer {layer_idx} local {updates+1}",
        )
        attempts += 1
        if not updated:
            continue
        updates += 1

        must_check = (
            updates == 1
            or updates % LOCAL_CHECK_EVERY == 0
            or updates == SCREEN_STEPS
            or updates == SCREEN_STEPS + POLISH_STEPS
            or updates == HARD_MAX_STEPS
        )
        if must_check:
            probe = local_metrics(repl, probe_cache)
            if probe["score"] < best["score"]:
                best = probe
                best_state = state_cpu(repl)
                best_step = updates
            print(
                f"    {updates:04d}/{HARD_MAX_STEPS} "
                f"NMSE={probe['nmse']:.4f} cos={probe['cosine']:.4f} "
                f"lr={core_lr:.6f} best={best['score']:.4f}"
            )

            # Once the 200-step screen passes, run only 100 polish updates.
            if updates == SCREEN_STEPS and local_gate(best):
                target_steps = SCREEN_STEPS + POLISH_STEPS
            # Easy layer finished after polish.
            if updates >= target_steps and local_gate(best):
                break

    repl.load_state_dict(best_state, strict=True)
    repl.eval()
    return {
        "local": best,
        "best_step": int(best_step),
        "updates": int(updates),
        "attempts": int(attempts),
        "passed": local_gate(best),
    }

def make_replacement(idx):
    return make_v28_full_replacement(
        teacher.encoder.layers[idx].attn,
        feature_dim=FULL_FEATURE_DIM,
        local_window=FULL_LOCAL_WINDOW,
        chunk_size=FULL_CHUNK_SIZE,
    ).to(device)

accepted = {}
history = []
completed = set()

def save_progress():
    torch.save({
        "version": "2.8",
        "full_layers": full_layers,
        "completed": sorted(completed),
        "accepted": accepted,
        "history": history,
        "student_state": {k: v.detach().cpu() for k, v in student.state_dict().items()},
    }, PROGRESS_PATH)

if RESUME and PROGRESS_PATH.exists():
    payload = torch.load(PROGRESS_PATH, map_location="cpu")
    if payload.get("version") != "2.8" or payload.get("full_layers") != full_layers:
        raise ValueError("V2.8 progress file is incompatible")
    for idx in payload.get("accepted", {}):
        i = int(idx)
        student.encoder.layers[i].attn = make_replacement(i)
    student.load_state_dict(payload["student_state"], strict=True)
    student.to(device).eval().requires_grad_(False)
    accepted = {int(k): v for k, v in payload.get("accepted", {}).items()}
    history = list(payload.get("history", []))
    completed = set(int(i) for i in payload.get("completed", []))
    print("resumed completed:", sorted(completed))

for idx in full_layers:
    if idx in completed:
        continue
    print("\n" + "=" * 100)
    print(f"V2.8 FAST FULL ATTENTION LAYER {idx}")
    print("=" * 100)

    train_cache = promote_layer_cache(train_cache_cpu[idx])
    probe_cache = promote_layer_cache(probe_cache_cpu[idx])
    repl = make_replacement(idx)

    result = train_local_fast(idx, repl, train_cache, probe_cache)
    print(json.dumps(result, indent=2))

    if not result["passed"]:
        history.append({"layer": idx, **result, "accepted": False})
        completed.add(idx)
        save_progress()
        raise RuntimeError(
            f"Layer {idx} failed the V2.8 local gate after "
            f"{result['updates']} updates. V2.8 requires ALL full-attention "
            "layers to pass before joint recovery."
        )

    student.encoder.layers[idx].attn = repl.eval().requires_grad_(False)
    accepted[idx] = {
        "replacement": "pdelta3_local32",
        "local": result["local"],
        "best_step": result["best_step"],
        "updates": result["updates"],
    }
    history.append({"layer": idx, **result, "accepted": True})
    completed.add(idx)
    save_progress()

    # Free GPU cache for this layer. CPU caches are no longer needed either.
    del train_cache, probe_cache, repl
    train_cache_cpu.pop(idx, None)
    probe_cache_cpu.pop(idx, None)
    torch.cuda.empty_cache()

if sorted(accepted) != full_layers:
    raise RuntimeError(
        f"V2.8 expected all full layers. Accepted={sorted(accepted)}, "
        f"expected={full_layers}"
    )

print("\n✅ ALL full-attention layers installed:", sorted(accepted))
print("Sliding attention remains native:", sliding_layers)


In [ ]:
#@title 7. One joint recovery for the complete all-full replacement model
def full_attn_forward(model, b, with_markers=True):
    captured = {}
    handles = []

    def make_hook(idx):
        def hook(mod, args, kwargs, out):
            captured[idx] = out[0]
        return hook

    for idx in full_layers:
        handles.append(
            model.encoder.layers[idx].attn.register_forward_hook(
                make_hook(idx), with_kwargs=True
            )
        )
    try:
        if with_markers:
            logits, actions, markers = forward_with_markers(model, b)
        else:
            logits, actions = model(**b)
            markers = None
    finally:
        for h in handles:
            h.remove()
    return logits, actions, markers, captured

def intermediate_alignment_loss(student_features, teacher_features, valid):
    losses = []
    for idx in full_layers:
        s = student_features[idx].float()
        t = teacher_features[idx].detach().float()
        m = valid[:, :, None].float()
        power = ((t * t) * m).sum() / (m.sum() * t.shape[-1]).clamp_min(1)
        nmse = (((s - t) ** 2) * m).sum() / (
            (m.sum() * s.shape[-1]).clamp_min(1) * power.clamp_min(1e-8)
        )
        cos = F.cosine_similarity(s[valid], t[valid], dim=-1).mean()
        losses.append(nmse + 0.50 * (1.0 - cos))
    return torch.stack(losses).mean()

def all_replacement_params():
    core, output = [], []
    seen = set()
    for idx in full_layers:
        module = student.encoder.layers[idx].attn
        c, o = split_v28_parameters(module)
        for p in c:
            if id(p) not in seen:
                seen.add(id(p)); core.append(p)
        for p in o:
            if id(p) not in seen:
                seen.add(id(p)); output.append(p)
    return core, output

def replacement_snapshot():
    return {
        idx: {
            k: v.detach().cpu().clone()
            for k, v in student.encoder.layers[idx].attn.state_dict().items()
        }
        for idx in full_layers
    }

def restore_snapshot(snapshot):
    for idx in full_layers:
        student.encoder.layers[idx].attn.load_state_dict(snapshot[idx], strict=True)

student.eval().requires_grad_(False)
before_joint = validate_pair(student, dev_items, DEV_GATE_LIMIT)
print("all-full candidate before joint recovery:", before_joint)

core, output = all_replacement_params()
for p in core + output:
    p.requires_grad = True
    if p.is_floating_point():
        p.data = p.data.float()

opt = torch.optim.AdamW(
    [
        {"params": core, "lr": 2.0e-4, "weight_decay": 1e-4},
        {"params": output, "lr": 2.0e-5, "weight_decay": 1e-4},
    ],
    betas=(0.9, 0.95),
)
scaler = torch.amp.GradScaler(
    "cuda", enabled=amp_dtype == torch.float16, init_scale=128.0
)

best_metrics = before_joint
best_score = dev_rank(before_joint)
best_state = replacement_snapshot()
best_step = 0
updates = attempts = 0
target_steps = JOINT_STEPS

while updates < JOINT_HARD_MAX_STEPS:
    core_lr = phase_lr(
        updates,
        JOINT_HARD_MAX_STEPS,
        2.0e-4,
        2.0e-5,
    )
    opt.param_groups[0]["lr"] = core_lr
    opt.param_groups[1]["lr"] = core_lr * 0.10

    b = batch(train_items, 9000 + attempts * JOINT_BATCH, JOINT_BATCH)
    valid = b["attention_mask"].bool()
    opt.zero_grad(set_to_none=True)

    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        tl, ta, tm, tf = full_attn_forward(teacher, b)

    # Keep replacements in eval mode: dropout is disabled and PDelta recurrent
    # state is not detached every chunk, while autograd still trains parameters.
    student.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        sl, sa, sm, sf = full_attn_forward(student, b)
        loss = decision_distill_loss(
            sl, tl, b["marker_mask"].bool(), sa, ta,
            temperature=1.25,
            rank_margin=0.15,
            rank_weight=0.15,
        )
        loss = loss + 0.05 * marker_alignment_loss(
            sm, tm, b["marker_mask"].bool()
        )
        loss = loss + 0.02 * intermediate_alignment_loss(
            sf, tf, valid
        )

    updated = guarded_step(
        loss, opt, scaler,
        [(core, 0.50), (output, 0.20)],
        context=f"V2.8 joint {updates+1}",
    )
    attempts += 1
    if not updated:
        continue
    updates += 1

    if (
        updates == 1
        or updates % JOINT_CHECK_EVERY == 0
        or updates == target_steps
        or updates == JOINT_HARD_MAX_STEPS
    ):
        metrics = validate_pair(student, dev_items, DEV_GATE_LIMIT)
        score = dev_rank(metrics)
        if score < best_score:
            best_metrics = metrics
            best_score = score
            best_state = replacement_snapshot()
            best_step = updates

        print(
            f"  joint {updates:03d}/{JOINT_HARD_MAX_STEPS} "
            f"loss={float(loss.detach()):.4f} "
            f"agree={metrics['agreement']:.4f} "
            f"JS={metrics['mean_js']:.5f} "
            f"gold_drop={metrics['accuracy_drop']:+.4f}"
        )

        if updates >= JOINT_STEPS and quality_pass(best_metrics):
            break

restore_snapshot(best_state)
student.eval().requires_grad_(False)
after_joint = validate_pair(student, dev_items, DEV_GATE_LIMIT)

joint_report = {
    "before": before_joint,
    "selected": after_joint,
    "best_step": int(best_step),
    "updates": int(updates),
    "attempts": int(attempts),
    "passed": quality_pass(after_joint),
}
print(json.dumps(joint_report, indent=2))

if not joint_report["passed"]:
    raise RuntimeError(
        "All 10 full-attention layers were replaced locally, but the complete "
        "V2.8 all-full model did not pass the joint teacher-quality gate. "
        "The best joint checkpoint is restored for analysis."
    )

save_progress()
print("✅ V2.8 all-full model passed the joint gate")


In [ ]:
#@title 8. Final untouched official-test evaluation + latency + checkpoint
test_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="test"))
test_items = make_items_from_rows(test_rows, SEED + 99, False)

dev_final = validate_pair(student, dev_items, len(dev_items))
test_final = validate_pair(student, test_items, len(test_items))

def benchmark_model(model, selected, warmup=5, repeats=20):
    b = direct_batch(selected)
    model.eval()
    for _ in range(warmup):
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    torch.cuda.synchronize()

    values = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
        torch.cuda.synchronize()
        values.append((time.perf_counter() - t0) * 1000.0)

    return {
        "median_ms": float(np.median(values)),
        "p95_ms": float(np.percentile(values, 95)),
        "mean_ms": float(np.mean(values)),
        "repeats": repeats,
    }

teacher_latency = benchmark_model(teacher, dev_items[:1])
student_latency = benchmark_model(student, dev_items[:1])
latency = {
    "teacher": teacher_latency,
    "student": student_latency,
    "speedup": (
        teacher_latency["median_ms"]
        / max(student_latency["median_ms"], 1e-9)
    ),
}

audit = {
    "full_layers": full_layers,
    "replaced_full_layers": [
        i for i in full_layers
        if isinstance(student.encoder.layers[i].attn, PDelta3GDN2CLVRAttention)
    ],
    "native_sliding_layers": sliding_layers,
}
audit["all_full_replaced"] = audit["replaced_full_layers"] == full_layers

report = {
    "version": "2.8",
    "source_model": SOURCE_MODEL,
    "protocol": "fast-full-attention-first",
    "architecture": {
        "full_attention": "PDelta3-GDN2 + global-linear-memory + Local32",
        "feature_dim": FULL_FEATURE_DIM,
        "local_window": FULL_LOCAL_WINDOW,
        "sliding_attention": "native ModernBERT sliding attention",
    },
    "speed_optimizations": {
        "multi_layer_teacher_cache": True,
        "teacher_cache_batches": TRAIN_CACHE_BATCHES,
        "screen_steps": SCREEN_STEPS,
        "polish_steps": POLISH_STEPS,
        "hard_max_steps": HARD_MAX_STEPS,
        "per_layer_e2e_recovery": False,
        "single_joint_recovery": True,
    },
    "audit": audit,
    "local_history": history,
    "joint_recovery": joint_report,
    "dev_final": dev_final,
    "official_test_final": test_final,
    "official_test_used_for_selection": False,
    "latency": latency,
}
REPORT_PATH.write_text(
    json.dumps(report, indent=2, allow_nan=False),
    encoding="utf-8",
)

checkpoint_path = OUTPUT_DIR / "v28_final.pt"
torch.save({
    "version": "2.8",
    "full_layers": full_layers,
    "state_dict": {k: v.detach().cpu() for k, v in student.state_dict().items()},
    "report": report,
}, checkpoint_path)

print("\n" + "=" * 100)
print("V2.8 FINAL")
print("=" * 100)
print("all full replaced:", audit["all_full_replaced"])
print("replaced full layers:", audit["replaced_full_layers"])
print("native sliding layers:", audit["native_sliding_layers"])
print("\nDEV:", json.dumps(dev_final, indent=2))
print("\nOFFICIAL TEST:", json.dumps(test_final, indent=2))
print("\nLATENCY:", json.dumps(latency, indent=2))
print("\nReport:", REPORT_PATH)
print("Checkpoint:", checkpoint_path)


In [ ]:
#@title 9. Reload sanity check
RUN_RELOAD_CHECK = True #@param {type:"boolean"}

if RUN_RELOAD_CHECK:
    payload = torch.load(OUTPUT_DIR / "v28_final.pt", map_location="cpu")
    reloaded, _ = build_checkpoint_model(source_dir)
    reloaded.to(device).eval().requires_grad_(False)
    reloaded.encoder.config.reference_compile = False

    for idx in full_layers:
        reloaded.encoder.layers[idx].attn = make_v28_full_replacement(
            reloaded.encoder.layers[idx].attn,
            feature_dim=FULL_FEATURE_DIM,
            local_window=FULL_LOCAL_WINDOW,
            chunk_size=FULL_CHUNK_SIZE,
        ).to(device)

    reloaded.load_state_dict(payload["state_dict"], strict=True)
    reloaded.eval().requires_grad_(False)

    sanity_items = dev_items[:min(32, len(dev_items))]
    current = validate_pair(student, sanity_items, len(sanity_items))
    restored = validate_pair(reloaded, sanity_items, len(sanity_items))
    print("current :", current)
    print("reloaded:", restored)

    if abs(current["agreement"] - restored["agreement"]) > 1e-12:
        raise RuntimeError("V2.8 reload sanity check failed")
    print("✅ V2.8 reload sanity check passed")
